# Jour 3 · Transformer des fonctions Python en tools


## Commencer sans agent

Un **tool** n'est pas une intelligence artificielle. C'est d'abord une fonction Python qui accomplit une tâche précise.

Nous écrivons et testons la fonction normalement avant de la rendre visible au modèle. Ainsi, une erreur métier reste reproductible sans LLM.


In [ ]:
def calculer_sla_python(priorite: str, utilisateurs_impactes: int) -> str:
    priorite_normalisee = priorite.strip().upper()
    if utilisateurs_impactes < 0:
        raise ValueError("Le nombre d'utilisateurs ne peut pas être négatif.")
    if priorite_normalisee == "CRITIQUE" or utilisateurs_impactes >= 250:
        return "SLA : 1 heure · escalade DSI requise"
    if priorite_normalisee == "HAUTE":
        return "SLA : 4 heures"
    if priorite_normalisee == "NORMALE":
        return "SLA : 1 jour ouvré"
    return "Priorité inconnue : validation humaine requise"


assert calculer_sla_python("CRITIQUE", 12).startswith("SLA : 1 heure")
assert calculer_sla_python("NORMALE", 1) == "SLA : 1 jour ouvré"
print(calculer_sla_python("HAUTE", 400))


## Le décorateur `@tool`

Le décorateur transforme la fonction en objet compréhensible par LangChain. Le code métier ne change pas : un nom, des types et une docstring sont ajoutés au contrat visible par le modèle.


![Fonction Python transformée en outil LangChain](../../assets/jour_01/10_fonction_vers_tool.png)

*Le décorateur expose un contrat ; la fonction continue d'exécuter le travail réel.*


In [ ]:
from langchain_core.tools import tool


@tool
def calculer_sla(priorite: str, utilisateurs_impactes: int) -> str:
    """Calcule le délai de prise en charge d'un ticket helpdesk.

    Args:
        priorite: Priorité déclarée : CRITIQUE, HAUTE ou NORMALE.
        utilisateurs_impactes: Nombre estimé d'utilisateurs bloqués.
    """
    return calculer_sla_python(priorite, utilisateurs_impactes)


print("Nom         :", calculer_sla.name)
print("Description :", calculer_sla.description)
print("Schéma      :", calculer_sla.args_schema.model_json_schema())


## Appeler le tool directement

Avant de laisser un agent décider, nous utilisons `.invoke()` nous-mêmes. L'entrée est un dictionnaire dont les clés correspondent aux arguments typés.


In [ ]:
resultat = calculer_sla.invoke({
    "priorite": "HAUTE",
    "utilisateurs_impactes": 400,
})
print(resultat)


## La docstring est le mode d'emploi du modèle

Le modèle ne choisit pas un tool en lisant tout son code. Il reçoit surtout son nom, la description et le schéma des arguments.

Une description doit dire quand utiliser l'outil et ce qu'il retourne, sans promettre une capacité absente.


![Nom, arguments et docstring composent le contrat](../../assets/jour_01/10_contrat_tool.png)

*Des types stricts et une description précise réduisent les appels incorrects.*


## Un deuxième outil très limité

Un outil étroit est plus facile à tester et à autoriser qu'une fonction « tout faire ».


In [ ]:
@tool
def astreinte_active(heure: int) -> str:
    """Indique si l'astreinte de nuit est active pour une heure entière entre 0 et 23."""
    if not 0 <= heure <= 23:
        return "Heure invalide : fournir une valeur entre 0 et 23."
    active = heure >= 18 or heure < 8
    return "Astreinte active" if active else "Astreinte inactive"


for heure in [7, 10, 20, 30]:
    print(heure, "→", astreinte_active.invoke({"heure": heure}))


### À vous de jouer — améliorer un contrat

Rendez le tool de sécurité non ambigu : arguments documentés, valeurs normalisées et réponse prudente pour une valeur inconnue.


In [ ]:
# Correction — contrat précis et valeurs validées
@tool
def verifier_secret(type_secret: str) -> str:
    """Indique si un type d'information peut être transmis au support.

    Args:
        type_secret: Type d'information, par exemple mot_de_passe, code_mfa ou identifiant.
    """
    secret = type_secret.strip().lower()
    if secret in {"mot_de_passe", "code_mfa"}:
        return "Interdit : ce secret ne doit jamais être transmis."
    if secret == "identifiant":
        return "Autorisé si nécessaire au diagnostic et sans autre donnée sensible."
    return "Type inconnu : demander une vérification humaine."


print(verifier_secret.invoke({"type_secret": "code_mfa"}))


## Définir la frontière de sécurité

Un tool peut lire une information, calculer une valeur ou déclencher une action. Plus l'action est difficile à annuler, plus les contrôles doivent être forts.

Dans ce cours, les tools restent locaux et sans effet externe : ils ne ferment aucun ticket, n'envoient aucun message et ne modifient aucun système d'entreprise.


## À retenir

- Tester la fonction avant le tool.
- Décrire précisément son rôle et ses arguments.
- Valider les entrées dans le code Python.
- Préférer plusieurs outils étroits à un outil vague.
- Ne pas donner d'action destructive à un agent sans contrôle supplémentaire.
